# Dask
This is a tutorial to use the Dask cluster from Jupyter, without Prefect.

In [4]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
init_demo()
init_dask_cluster_staging(scale=2)
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/0cf6c1ffe71c4267a0cf508bac616536/status
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/60dccff86a1e496aaf710dc397bd74dd/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| msgpack | 1.1.0  | 1.0.7     | 1.0.7   |
| pandas  | None   | 2.2.3     | 2.2.3   |
| toolz   | 1.0.0  | 0.12.0    | 0.12.0  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.1.2  | 1.26.4    | 1.26.4  |
| pandas  | None   | 2.2.3     | 2.2.3   |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMisma

In [5]:
# Other imports
import logging
import os
import sys
from pathlib import Path

In [6]:
# My local "./resources" folder contains a "dask_utils.py" module.
# I want to be able to use the same "import dask_utils" line on both client and workers.
# For this, I'm updating my PYTHONPATH.
sys.path.append("./resources")
import dask_utils

# Show my client IP address using a function from this local module
logging.warning(f" Client IP address: {dask_utils.get_ip_address()}")

# I want to be able to do the same from the dask workers. 
for client in dask_client_staging, dask_client_eopf:
    
    # First I need to forward logging from dask workers to the client.
    # NOTE: we need to the the logging in the workers, "print" won't be forwarded.
    client.forward_logging()
    
    # Then I need to upload my local module to the dask workers
    client.upload_file("./resources/dask_utils.py")

## 1. Implement the `Futures` tutorial: https://docs.dask.org/en/stable/futures.html
Note: this is how the `rs-server-staging` web service is using Dask.

In [7]:
def inc(x, name):

    # From staging workers
    if name == "staging":
        # Just make sure that rs-server-staging is installed inside the dask workers.
        # NOTE: this import doesn't run the staging web service. 
        # It only imports its modules to be able to call the staging functions.
        # This is actually what the staging web service (that runs on another pod) is doing.
        from rs_server_staging.processors import processors

    # From eopf workers, we can also import the eopf modules
    else:
        from eopf.product.eo_product import EOProduct        
    
    # Note that this is run from a dask worker with a different IP than the client,
    # and that the workers also differ between the staging and eopf workers.
    logging.warning(f" Worker IP address for {name!r}: {dask_utils.get_ip_address()}")

    return x + 1

def add(x, y):
    return x + y

# Test this for the staging and eopf client
for client, name in ((dask_client_staging, "staging"), (dask_client_eopf, "eopf")):
    print(f"\nTest {name!r}:")
    
    a = client.submit(inc, 10, name)  # calls inc(10) in background thread or process
    b = client.submit(inc, 20, name)  # calls inc(20) in background thread or process
    print(f"a: {a.result()}")
    print(f"b: {b.result()}")

    c = client.submit(add, a, b)  # calls add on the results of a and b
    print(f"c: {c.result()}")

    futures = client.map(inc, range(5), name=name)
    results = client.gather(futures)  # this can be faster
    print(results)


Test 'staging':


a: 11
b: 21
c: 32
[1, 2, 3, 4, 5]

Test 'eopf':


a: 11
b: 21
c: 32
[1, 2, 3, 4, 5]


## 2. `pip install` inside Dask workers

In [8]:
# Test with any cluster
client = dask_client_eopf

In [9]:
# Test if a module is installed inside the dask workers
def test_pip():
    import argh # yes this is a real module, see: https://pypi.org/project/argh/
    logging.warning(f" argh methods/attributes: {dir(argh)}")

# The first time you will test this in workers, it will fail
try:
    client.submit(test_pip).result()
except ModuleNotFoundError:
    print("'argh' is not yet installed in the workers ...")

# You can install it with: https://distributed.dask.org/en/stable/plugins.html#built-in-scheduler-plugins
from dask.distributed import PipInstall
plugin = PipInstall(packages=["argh"])
client.register_plugin(plugin)

# Now it will work.
client.submit(test_pip, pure=False).result() # IMPORTANT: use pure=False to disable cache
print("'argh' is now installed in the workers.")

'argh' is not yet installed in the workers ...


'argh' is now installed in the workers.


In [10]:
# Do the same with a wheel file. First download it.
whl_dir = "/tmp/emoji"
!rm -rf $whl_dir && mkdir -p $whl_dir && pip download --dest $whl_dir emoji
whl_file = os.listdir(whl_dir)[0]
whl_path = Path(whl_dir) / whl_file

# Then we'll upload and install it in the dask workers. 
# But it only works with .py, .egg or .zip
# See: https://distributed.dask.org/en/latest/api.html#distributed.Client.upload_file
# A .whl file is just a zip, so rename it.
whl_path = whl_path.rename(whl_path.with_suffix(".zip"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 16.1 MB/s eta 0:00:00
Saved /tmp/emoji/emoji-2.14.1-py3-none-any.whl
Successfully downloaded emoji

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip


In [11]:
# Then do the same as before
def test_pip_whl():
    import emoji
    logging.warning(f" emoji methods/attributes: {dir(emoji)}")
try:
    client.submit(test_pip_whl, pure=False).result()
except ModuleNotFoundError:
    print("'emoji' is not yet installed in the workers ...")

# Upload the wheel/zip. It is automatically installed.
client.upload_file(str(whl_path))

client.submit(test_pip_whl, pure=False).result() # IMPORTANT: use pure=False to disable cache
print("'emoji' is now installed in the workers.")

'emoji' is not yet installed in the workers ...


'emoji' is now installed in the workers.


## 3. Shutdown the dask clusters

In [12]:
# You can scale the clusters to 0 workers
dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_staging)
shutdown_dask_clusters(dask_gateway_eopf)

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

Shutting down cluster '0cf6c1ffe71c4267a0cf508bac616536' ...
Shutting down cluster '60dccff86a1e496aaf710dc397bd74dd' ...


2025-01-29 10:33:43,214 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client
2025-01-29 10:33:43,216 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client
2025-01-29 10:33:43,241 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client
2025-01-29 10:33:43,244 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client
